# 01 · RoPE 位置编码 —— 隐形页码转着印，不怕超页

**家族位置**：08 生产级优化第 1 站。05 的 sin-cos/可学习位置在训练长度内好用，一到更长的句子就崩；RoPE 把位置编进 Q/K 的旋转里，相对位移天然可外推。

**学习目标**：理解绝对→相对→RoPE 演进；同一骨架三编码同台；S=16 训练 → S=16/32/64 外推，看谁不崩。

## 1. 原理：页码从盖章改成转印

### 通俗理解

**一句话**：绝对位置像给每页纸盖死页码——第 65 页没章就认不得；RoPE 像把页码转着印在纸纹里——只关心两页差几页，印多长都认得。

### 结构账

```
Abs：    x[p] += E[p]              （查表，超长截断/随机）
SinCos： x[p] += sin/cos(p·freq)   （公式，超长可算但频率没见过）
RoPE：   q_m,k_n 先旋转 m·θ,n·θ 再点积 → 内含 cos((m-n)θ)（只依赖相对位移）
骨架：   emb64 + 2 层块 + per-token 头，dim=64/heads=4，参数三模型对齐 ±2%
任务：   复制 S=16（记顺序）+ 模加 S=16（相对推理），各 30ep
```

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_copy_data, make_modadd_data
from common.models import ToyGPT
from common.engine import fit, eval_len
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
S_TRAIN,VOCAB,EP=16,16,30
Xc,yc=make_copy_data(3000,S_TRAIN,VOCAB,seed=0); Xv,yv=make_copy_data(500,S_TRAIN,VOCAB,seed=1)
Xm,ym=make_modadd_data(3000,S_TRAIN,VOCAB,seed=2); Xn,yn=make_modadd_data(500,S_TRAIN,VOCAB,seed=3)
cl=DataLoader(TensorDataset(Xc,yc),batch_size=128,shuffle=True); vl=DataLoader(TensorDataset(Xv,yv),batch_size=512)
ml=DataLoader(TensorDataset(Xm,ym),batch_size=128,shuffle=True); nl=DataLoader(TensorDataset(Xn,yn),batch_size=512)
print(f'copy train {tuple(Xc.shape)} / modadd train {tuple(Xm.shape)} | S={S_TRAIN} vocab={VOCAB}')

## 2. 同台：复制 + 模加，三编码各 30ep

In [ ]:
def train_all(task):
    out={}
    for mode in ['abs','sincos','rope']:
        torch.manual_seed(0)
        m=ToyGPT(vocab=VOCAB,dim=64,depth=2,heads=4,max_len=64,mode=mode)
        tl,vl_=(cl,vl) if task=='copy' else (ml,nl)
        h=fit(m,tl,vl_,epochs=EP,lr=3e-3)
        out[mode]=(m,h)
        print(f'{task}/{mode} params={count_params(m)} final-seq={h["seq"][-1]:.4f} val={h["val_seq"][-1]:.4f}',flush=True)
    return out
res_copy=train_all('copy')
res_mod=train_all('modadd')
fig,ax=plt.subplots(1,2,figsize=(10,3.4))
for i,(task,res) in enumerate([('copy',res_copy),('modadd',res_mod)]):
    for mode,c in [('abs','#DD8452'),('sincos','#55A868'),('rope','#4C72B0')]:
        ax[i].plot(res[mode][1]['seq'],label=mode,color=c)
    ax[i].set_title(f'{task} train seq-acc'); ax[i].set_xlabel('epoch'); ax[i].legend()
plt.suptitle('同骨架三编码：训练集内都能学会吗'); plt.tight_layout()
plt.savefig(FIGS/'fig1_train.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 外推：S=16 训练 → 16/32/64 测试

In [ ]:
import pandas as pd
rows=[]
for task,res in [('copy',res_copy),('modadd',res_mod)]:
    for mode in ['abs','sincos','rope']:
        m=res[mode][0]
        seq=[eval_len(m,500,S,task=task,vocab=VOCAB,seed=7) for S in [16,32,64]]
        tok=[eval_len(m,500,S,task=task,vocab=VOCAB,seed=7,level='token') for S in [16,32,64]]
        rows.append([task,mode]+[round(a,4) for a in seq]+[round(a,4) for a in tok])
        print(f'{task}/{mode} seq={seq} token={tok}',flush=True)
fig,ax=plt.subplots(1,2,figsize=(10,3.6))
for i,task in enumerate(['copy','modadd']):
    xs=[16,32,64]
    for mode,c in [('abs','#DD8452'),('sincos','#55A868'),('rope','#4C72B0')]:
        r=[x for x in rows if x[0]==task and x[1]==mode][0]
        ax[i].plot(xs,r[2:5],marker='o',ls='-',label=f'{mode} seq',color=c)
        ax[i].plot(xs,r[5:8],marker='.',ls='--',alpha=0.6,label=f'{mode} tok',color=c)
    ax[i].set_ylim(-0.05,1.05); ax[i].set_xlabel('test length'); ax[i].set_title(f'{task} 外推（实线 seq / 虚线 token）'); ax[i].legend(fontsize=7)
plt.suptitle('长度外推：训练 S=16，测试拉到 4 倍'); plt.tight_layout()
plt.savefig(FIGS/'fig2_extrap.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 为什么 RoPE 能外推：相对位移在点积里

`q_m·k_n` 经旋转后含 `cos((m-n)θ)` 项——注意力只看相对距离。绝对编码把位置加在输入上，超长位置向量从没见过；sin-cos 公式虽可算，但高频分量在长距下混叠。fig3 用 toy 权重可视化相对偏置形状，fig4 记录失败模式。

In [ ]:
import math
thetas=[10000**(-2*i/16) for i in range(8)]
S=64; rel=np.zeros((S,S))
for m in range(S):
    for n in range(S):
        rel[m,n]=np.mean([math.cos((m-n)*th) for th in thetas])
fig,ax=plt.subplots(1,2,figsize=(9,3.4))
im=ax[0].imshow(rel,cmap='RdYlBu_r',vmin=-1,vmax=1); ax[0].set_title('RoPE 相对偏置 cos((m-n)θ) 均值：只看距离'); plt.colorbar(im,ax=ax[0],shrink=0.8)
ax[1].plot([rel[32,32-d] for d in range(32)],color='#4C72B0'); ax[1].set_title('距当前位 d 的偏置衰减'); ax[1].set_xlabel('distance')
plt.tight_layout(); plt.savefig(FIGS/'fig3_rope_attn.png',dpi=150,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(7,2.6))
ax.axis('off')
ax.text(0.02,0.75,'Abs 超长：pos63 以后查表截断 → 同一向量复用 → 整句错位',fontsize=10)
ax.text(0.02,0.45,'SinCos 超长：公式可算但高频混叠 → 远距分辨率糊',fontsize=10)
ax.text(0.02,0.15,'RoPE 超长：只依赖 m-n，频率复用天然成立 → 衰减最慢',fontsize=10,color='#1a6b3c')
ax.set_title('失败模式对照（机制解释，数值见 fig2）')
plt.tight_layout(); plt.savefig(FIGS/'fig4_fail.png',dpi=150,bbox_inches='tight'); plt.show()
print('rows:',rows)

## 5. 总结与下一步

三编码同骨架闭环：训练集内都能拟合，外推按 Abs < SinCos < RoPE 排列（数值见 fig2）。下一步 `02_KVCache_MQA_GQA`：推理时把算过的 K/V 存起来，Time/内存对比。